# 03 — Build and Train the Deterministic GRU World Model

This notebook traces the exact shape flow through the model and performs a tiny training run.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

In [2]:
import torch
from torch.utils.data import DataLoader
from apexsim.config import load_config
from apexsim.contracts import MODEL_INPUT_COLUMNS, TARGET_COLUMNS
from apexsim.data.features import Standardizer
from apexsim.data.windows import TelemetryWindowDataset
from apexsim.models.gru_world_model import GRUWorldModel

config = load_config(ROOT/'configs/fast.yaml')
frame = pd.read_csv(ROOT/'artifacts/runs/reference_gru/canonical_telemetry.csv')
splits = json.loads((ROOT/'artifacts/runs/reference_gru/splits.json').read_text())
scaler = Standardizer.from_dict(json.loads((ROOT/'artifacts/runs/reference_gru/standardizer.json').read_text()))
train_ds = TelemetryWindowDataset(frame, splits['train'], 16, 4, scaler, max_windows=128)
loader = DataLoader(train_ds, batch_size=16, shuffle=True)
model = GRUWorldModel(len(MODEL_INPUT_COLUMNS), len(TARGET_COLUMNS), hidden_dim=32)
model

GRUWorldModel(
  (encoder): GRU(16, 32, batch_first=True)
  (transition): GRUCell(16, 32)
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): SiLU()
    (2): Linear(in_features=32, out_features=5, bias=True)
  )
)

## Shape story

- `history`: `[batch, history_time, 16]`
- GRU encoder compresses it to a hidden memory.
- At each future step, the model takes the previous predicted state plus the planned action/context.
- Decoder outputs five normalized next-state values.
- Its prediction becomes part of the next input, making rollout genuinely autoregressive.

In [3]:
batch = next(iter(loader))
with torch.no_grad():
    prediction = model(batch['history'], batch['future_inputs'])
print('history', batch['history'].shape)
print('future inputs', batch['future_inputs'].shape)
print('prediction', prediction.shape)
print('target', batch['future_targets'].shape)

history torch.Size([16, 16, 16])
future inputs torch.Size([16, 4, 16])
prediction torch.Size([16, 4, 5])
target torch.Size([16, 4, 5])


## One tiny training loop
Read it as English:

1. Forget stale gradients.
2. Imagine the future.
3. Compare imagination with the recorded future.
4. Compute which parameters caused the error.
5. Move parameters slightly toward a smaller error.

In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
losses=[]
for batch_index, batch in enumerate(loader):
    optimizer.zero_grad(set_to_none=True)
    predicted = model(batch['history'], batch['future_inputs'])
    loss = torch.nn.functional.smooth_l1_loss(predicted, batch['future_targets'])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(float(loss.detach()))
print('first loss', losses[0], 'last loss', losses[-1])

first loss 0.5091940760612488 last loss 0.42494696378707886


### Debug ritual
Before a long run, overfit 16–64 windows. If the training loss cannot collapse, suspect shape, normalization, target alignment, optimizer, masking or model code before blaming data volume.